# Command Following — Motor ERD Analysis

Measures volitional motor imagery via **Event-Related Desynchronization (ERD)**.  
A patient following the command "keep opening and closing your hand" should show
**power decreases in mu (8–12 Hz) and beta (14–30 Hz)** at contralateral motor electrodes
compared to the rest (stop) condition.

| Condition | Instruction | Expected response |
| --- | --- | --- |
| Keep | "Keep moving" | ERD at contralateral C3/C4 |
| Stop | "Stop moving" | Baseline / ERS (event-related synchronization) |

**Schema note:**
- CON010/013: `right_command` / `left_command` — run-level rows, 8 keep+stop cycles reconstructed
- CON011/012: `right_command+p` / `left_command+p` — same, with prompt

A **positive finding** is significant ERD (p < 0.05) in mu and/or beta at the contralateral motor
channel, with lateralization matching the commanded hand.

**Environment:** `stimulus_software/.venv` (MNE 1.11, scipy, pandas, matplotlib)

## 1. Configuration

In [ ]:
import os
import sys
from pathlib import Path

SUBJECT_ID   = 'CON013'
SESSION_DATE = '2026-04-10'

ANALYSIS_ROOT = Path.cwd().resolve()
if ANALYSIS_ROOT.name != 'analysis':
    ANALYSIS_ROOT = next(
        (parent for parent in [ANALYSIS_ROOT, *ANALYSIS_ROOT.parents]
         if parent.name == 'analysis' and (parent / 'notebooks').exists()),
        ANALYSIS_ROOT,
    )
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))

from lib.io import DEFAULT_EEG_CHANNELS, build_subject_paths

paths = build_subject_paths(SUBJECT_ID, SESSION_DATE, analysis_root=ANALYSIS_ROOT)
EDF_PATH = paths['edf_path']
CSV_PATH = paths['csv_path']
OUT_DIR = str(paths['out_dir'])
EEG_CHANNELS = DEFAULT_EEG_CHANNELS.copy()
BAD_CHANNELS = []  # e.g. ['Fp1', 'T3']

print(f'Subject: {SUBJECT_ID}  |  EDF: {EDF_PATH}')

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from scipy import stats

from lib.io import align_stimulus_csv, load_raw_eeg_metadata
from lib.preprocessing import load_filtered_eeg

%matplotlib inline
mne.set_log_level('WARNING')
print(f'MNE {mne.__version__}')

## 3. Load EDF + Sync Alignment

In [ ]:
raw, sfreq, available_eeg = load_raw_eeg_metadata(
    EDF_PATH,
    eeg_channels=EEG_CHANNELS,
    bad_channels=BAD_CHANNELS,
    preload=False,
    verbose=False,
)

print(f'EEG channels ({len(available_eeg)}): {available_eeg}')
print(f'sfreq: {sfreq} Hz  |  duration: {raw.n_times/sfreq:.0f}s')

In [ ]:
df, time_offset = align_stimulus_csv(CSV_PATH, sfreq=sfreq, n_times=raw.n_times)
print(f'time_offset = {time_offset:.4f} s')

## 4. Preprocessing

In [ ]:
# 1–40 Hz for motor bands (mu 8–12 Hz, beta 14–30 Hz)
raw_erd = load_filtered_eeg(raw, available_eeg, l_freq=1, h_freq=40, verbose=False)
print('Preprocessing done.')

## 5. Reconstruct Keep / Stop Epochs

Command rows are **run-level** (one row per 8-cycle run). Individual keep/stop pair timings
are reconstructed using known stimulus constants.

In [ ]:
cmd_df = df[df['stim_type'].str.contains('command', na=False)].copy()
cmd_df['side']       = cmd_df['stim_type'].str.extract(r'(right|left)')
cmd_df['has_prompt'] = cmd_df['stim_type'].str.contains(r'\+p', na=False)
print(f'Command runs: {len(cmd_df)}')
print(cmd_df[['stim_type', 'side', 'has_prompt', 'edf_start', 'edf_end']].to_string())

In [ ]:
# Timing constants (must match stimulus_software lib/constants.py)
KEEP_PAUSE_S   = 10.0   # keep imagery window length
STOP_PAUSE_S   = 10.0   # stop rest window length
TOTAL_CYCLES   = 8
PROMPT_DUR_EST = 4.0    # prompt audio + PROMPT_DELAY

run_durs   = (cmd_df['edf_end'] - cmd_df['edf_start']).values
has_prompt = cmd_df['has_prompt'].values

keep_events, stop_events = [], []
keep_meta,   stop_meta   = [], []

for i, (_, run) in enumerate(cmd_df.iterrows()):
    effective_dur    = run_durs[i] - (PROMPT_DUR_EST if has_prompt[i] else 0)
    audio_per_cycle  = max((effective_dur / TOTAL_CYCLES) - KEEP_PAUSE_S - STOP_PAUSE_S, 1.5)
    keep_audio_est   = audio_per_cycle / 2
    side             = run['side']

    t = run['edf_start'] + (PROMPT_DUR_EST if has_prompt[i] else 0)
    for cycle in range(TOTAL_CYCLES):
        keep_events.append([int(t * sfreq), 0, 1])
        keep_meta.append({'side': side, 'cycle': cycle, 'run': i})

        stop_t = t + keep_audio_est + KEEP_PAUSE_S
        stop_events.append([int(stop_t * sfreq), 0, 2])
        stop_meta.append({'side': side, 'cycle': cycle, 'run': i})

        t = stop_t + keep_audio_est + STOP_PAUSE_S

keep_events = np.array(keep_events, dtype=int)
stop_events = np.array(stop_events, dtype=int)
keep_meta_df = pd.DataFrame(keep_meta)
stop_meta_df = pd.DataFrame(stop_meta)

print(f'Keep events: {len(keep_events)}')
print(f'Stop events: {len(stop_events)}')
print(keep_meta_df.groupby('side').size().rename('cycles').to_string())

In [ ]:
MOTOR_CHANNELS = [ch for ch in ['C3', 'Cz', 'C4'] if ch in available_eeg]
print(f'Motor channels: {MOTOR_CHANNELS}')

all_cmd_events = np.vstack([keep_events, stop_events])
all_cmd_events = all_cmd_events[all_cmd_events[:, 0].argsort()]

epochs_cmd = mne.Epochs(
    raw_erd, events=all_cmd_events,
    event_id={'keep': 1, 'stop': 2},
    tmin=0, tmax=9.9, baseline=None,
    preload=True, verbose=False
)
print(f'Epochs: {len(epochs_cmd)}  ({len(epochs_cmd["keep"])} keep, {len(epochs_cmd["stop"])} stop)')

## 6. Compute ERD

In [ ]:
keep_ep = epochs_cmd['keep'].copy().pick(MOTOR_CHANNELS)
stop_ep = epochs_cmd['stop'].copy().pick(MOTOR_CHANNELS)

psd_k = keep_ep.compute_psd(method='welch', fmin=1, fmax=40, verbose=False)
psd_s = stop_ep.compute_psd(method='welch', fmin=1, fmax=40, verbose=False)
data_k, freqs_psd = psd_k.get_data(return_freqs=True)
data_s, _          = psd_s.get_data(return_freqs=True)

# Average across channels for overall ERD spectrum
keep_avg = data_k.mean(axis=(0, 1))
stop_avg = data_s.mean(axis=(0, 1))
erd_db   = 10 * np.log10(keep_avg / (stop_avg + 1e-30))

# Band-average ERD per epoch for statistics
MU_BAND   = (8,  12)
BETA_BAND = (14, 30)

def band_power(data, freqs, fmin, fmax):
    """Mean power across frequency band. data: (n_trials, n_ch, n_freqs)"""
    mask = (freqs >= fmin) & (freqs <= fmax)
    return data[:, :, mask].mean(axis=(1, 2))  # (n_trials,)

mu_k   = band_power(data_k, freqs_psd, *MU_BAND)
mu_s   = band_power(data_s, freqs_psd, *MU_BAND)
beta_k = band_power(data_k, freqs_psd, *BETA_BAND)
beta_s = band_power(data_s, freqs_psd, *BETA_BAND)

print(f'Mu    keep: {mu_k.mean():.2e}   stop: {mu_s.mean():.2e}')
print(f'Beta  keep: {beta_k.mean():.2e}  stop: {beta_s.mean():.2e}')

## 7. Statistical Test

In [ ]:
# Paired t-test: keep < stop would indicate ERD (motor imagery engaging motor cortex)
# One-sided: keep power LESS than stop power => t < 0, p_erd < 0.05
t_mu,   p_mu_two   = stats.ttest_rel(mu_k,   mu_s)
t_beta, p_beta_two = stats.ttest_rel(beta_k, beta_s)

# One-sided p (keep < stop)
p_mu   = p_mu_two   / 2 if t_mu   < 0 else 1 - p_mu_two   / 2
p_beta = p_beta_two / 2 if t_beta < 0 else 1 - p_beta_two / 2

# Cohen's d for effect size
def cohens_d_paired(a, b):
    diff = a - b
    return diff.mean() / (diff.std() + 1e-30)

d_mu   = cohens_d_paired(mu_k,   mu_s)
d_beta = cohens_d_paired(beta_k, beta_s)

print(f'{"Band":<8} {"Keep mean":>12} {"Stop mean":>12} {"t":>8} {"p (1-sided)":>12} {"d":>8} {"Sig":>5}')
print("-" * 65)
for band, pm, t, p, d in [("Mu", "8–12 Hz", t_mu, p_mu, d_mu), ("Beta", "14–30 Hz", t_beta, p_beta, d_beta)]:
    sig = "✓" if p < 0.05 else ""
    print(f'{band:<8} {pm:>12} {"":>12} {t:>8.3f} {p:>12.4f} {d:>8.3f} {sig:>5}')

## 8. Plots

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7))

axes[0].semilogy(freqs_psd, keep_avg * 1e12, color='steelblue', lw=1.5, label='Keep (motor imagery)')
axes[0].semilogy(freqs_psd, stop_avg * 1e12, color='firebrick',  lw=1.5, label='Stop (rest)')
for f0, f1, c, lbl in [(8, 12, 'gold', 'Mu'), (14, 30, 'lightgreen', 'Beta')]:
    axes[0].axvspan(f0, f1, color=c, alpha=0.2, label=lbl)
axes[0].set(xlabel='Hz', ylabel='PSD (pV²/Hz)',
            title=f'{SUBJECT_ID} — Motor channels ({MOTOR_CHANNELS}): Keep vs Stop PSD')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(freqs_psd, erd_db, color='purple', lw=1.5)
axes[1].axhline(0, color='k', lw=0.8, ls='--')
for f0, f1, c, lbl in [(8, 12, 'gold', f'Mu  p={p_mu:.3f}{"✓" if p_mu<0.05 else ""}'),
                        (14, 30, 'lightgreen', f'Beta  p={p_beta:.3f}{"✓" if p_beta<0.05 else ""}')]:
    axes[1].axvspan(f0, f1, color=c, alpha=0.2, label=lbl)
axes[1].set(xlabel='Hz', ylabel='Keep − Stop (dB)',
            title='ERD: negative = power decrease during Keep (expected for motor imagery)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'{SUBJECT_ID}_command_erd.png'), dpi=150)
plt.show()

In [ ]:
# Lateralization: per-side ERD at each motor channel
if 'C3' in available_eeg and 'C4' in available_eeg:
    for side in ['right', 'left']:
        contra = 'C3' if side == 'right' else 'C4'
        ipsi   = 'C4' if side == 'right' else 'C3'

        side_keep = keep_meta_df['side'] == side
        side_stop = stop_meta_df['side'] == side
        if side_keep.sum() == 0:
            continue

        ep_k = mne.Epochs(raw_erd, keep_events[side_keep.values], event_id={'keep': 1},
                          tmin=0, tmax=9.9, baseline=None, preload=True, verbose=False)
        ep_s = mne.Epochs(raw_erd, stop_events[side_stop.values], event_id={'stop': 1},
                          tmin=0, tmax=9.9, baseline=None, preload=True, verbose=False)

        fig, axes = plt.subplots(1, len(MOTOR_CHANNELS), figsize=(4 * len(MOTOR_CHANNELS), 4))
        if len(MOTOR_CHANNELS) == 1:
            axes = [axes]

        for ax, ch in zip(axes, MOTOR_CHANNELS):
            pk = ep_k.copy().pick([ch]).compute_psd(method='welch', fmin=1, fmax=40, verbose=False)
            ps = ep_s.copy().pick([ch]).compute_psd(method='welch', fmin=1, fmax=40, verbose=False)
            dk, f = pk.get_data(return_freqs=True)
            ds, _ = ps.get_data(return_freqs=True)
            erd_ch = 10 * np.log10(dk.mean(axis=(0,1)) / (ds.mean(axis=(0,1)) + 1e-30))
            ax.plot(f, erd_ch, color='purple', lw=1.5)
            ax.axhline(0, color='k', lw=0.8, ls='--')
            for f0, f1, c in [(8, 12, 'gold'), (14, 30, 'lightgreen')]:
                ax.axvspan(f0, f1, color=c, alpha=0.2)
            label = f'{ch}'
            if ch == contra: label += ' (contra ← expected ERD)'
            elif ch == ipsi:  label += ' (ipsi)'
            ax.set(title=label, xlabel='Hz', ylabel='Keep−Stop (dB)')
            ax.grid(True, alpha=0.3)

        fig.suptitle(f'{SUBJECT_ID} — {side.capitalize()} command ERD', fontsize=11)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f'{SUBJECT_ID}_command_{side}_lateralization.png'), dpi=150)
        plt.show()
else:
    print("C3 and/or C4 not available — skipping lateralization plot.")

## 9. Claassen SVM (requires scikit-learn)

The full clinical protocol (Claassen et al. 2019) uses an SVM classifier (LinearSVC + LOGO
cross-validation) to decode keep vs stop from all-channel spectral features, reporting AUC
with a permutation-test p-value.

Install scikit-learn in the venv, then run:
```bash
pip install scikit-learn
```

Reference implementation: `eeg-analysis/claassen_analysis.py` and
`awaken-ai/src/pipelines/command_following_claassen.py`.

## 10. Summary

**Positive finding:** ERD (p < 0.05, Cohen's d > 0.5) in mu and/or beta at contralateral motor
channel (C3 for right-hand command, C4 for left-hand command).  
**Clinical interpretation:** Volitional motor imagery in response to auditory command —
evidence of cognitive-motor dissociation (CMD).